# 04 · Sales plan 2027 – tables for Fabric Plan

Prepares a small, separate set of tables for the **2027 sales plan** built with *Plan* (Planning in Microsoft Fabric IQ).
The planning model `sm_sales_plan` is kept apart from the FP&A model `sm_fpa`, because a Plan item stays bound to one semantic model.

**Inputs** (upload `data/raw/plan` to `Files/raw/plan`): FX scenarios, production capacity, opening stock 1 Jan 2027, plan seed (2025 volumes +3 %, prices +2 %, currency mix).

**Outputs:**
* `plan_dim_month` – months 2025-01 … 2027-12 (actuals + plan year)
* `plan_dim_currency`, `plan_dim_scenario`
* `plan_fact_sales_actual` – monthly actual sales by product × market × **invoice currency** (from `fact_sales`)
* `plan_fx`, `plan_capacity`, `plan_opening_stock`, `plan_seed`

Currency is a separate dimension from market: an export customer may pay in PLN and a domestic one in EUR.


In [ ]:
PLAN_YEAR = 2027
RAW = "Files/raw/plan"


In [ ]:
from pyspark.sql import functions as F


def save_table(df, name: str) -> None:
    (df.write.mode("overwrite").format("delta")
        .option("overwriteSchema", "true")
        .saveAsTable(name))
    print(f"✔ {name}: {spark.read.table(name).count():,} rows")


def read_csv(file: str):
    return spark.read.option("header", True).option("inferSchema", True).csv(f"{RAW}/{file}")


## 1. Dimensions

In [ ]:
months = (spark.createDataFrame([("2025-01-01", f"{PLAN_YEAR}-12-01")], ["d0", "d1"])
    .select(F.explode(F.sequence(F.to_date("d0"), F.to_date("d1"), F.expr("interval 1 month"))).alias("month_start"))
    .withColumn("year", F.year("month_start"))
    .withColumn("quarter", F.concat(F.lit("Q"), F.quarter("month_start")))
    .withColumn("month_no", F.month("month_start"))
    .withColumn("month_name", F.date_format("month_start", "MMM"))
    .withColumn("year_month", F.date_format("month_start", "yyyy-MM"))
    .withColumn("period_type", F.when(F.col("year") == PLAN_YEAR, "Plan").otherwise("Actual")))
save_table(months, "plan_dim_month")

save_table(spark.createDataFrame([("PLN", 1), ("EUR", 2)], ["currency", "sort_order"]), "plan_dim_currency")

fx = (read_csv("fx_scenarios_2027.csv")
      .withColumn("month_start", F.to_date("month_start"))
      .withColumn("eur_pln", F.col("eur_pln").cast("decimal(10,4)")))
save_table(fx.select("scenario").distinct(), "plan_dim_scenario")
save_table(fx, "plan_fx")


## 2. Actual sales by month, product, market and invoice currency

In [ ]:
actual = (spark.read.table("fact_sales")
    .groupBy(F.trunc("invoice_date", "month").alias("month_start"), "product_id", "market", "currency")
    .agg(F.round(F.sum("quantity_t"), 1).alias("quantity_t"),
         F.round(F.sum("net_amount_cur"), 2).alias("revenue_cur"),
         F.round(F.sum("net_amount_pln"), 2).alias("revenue_pln")))
save_table(actual, "plan_fact_sales_actual")


## 3. Plan inputs: capacity, opening stock, seed

In [ ]:
save_table(read_csv("production_capacity_2027.csv").withColumn("month_start", F.to_date("month_start")), "plan_capacity")
save_table(read_csv("opening_stock_2027.csv"), "plan_opening_stock")

seed = (read_csv("plan_seed_2027.csv")
        .withColumn("month_start", F.to_date("month_start"))
        .withColumn("price_cur", F.col("price_cur").cast("decimal(18,2)")))
save_table(seed, "plan_seed")


## 4. Check: is the seed plan feasible? (closing stock with production = capacity)

In [ ]:
from pyspark.sql import Window

flow = (spark.read.table("plan_seed").groupBy("product_id", "month_start").agg(F.sum("quantity_t").alias("sales_t"))
        .join(spark.read.table("plan_capacity"), ["product_id", "month_start"])
        .withColumn("net_t", F.col("capacity_t") - F.col("sales_t")))
w = Window.partitionBy("product_id").orderBy("month_start").rowsBetween(Window.unboundedPreceding, 0)
check = (flow.withColumn("cum_net", F.sum("net_t").over(w))
         .join(spark.read.table("plan_opening_stock"), "product_id")
         .withColumn("closing_stock_t", F.col("opening_stock_t") + F.col("cum_net"))
         .groupBy("month_start").pivot("product_id").agg(F.round(F.first("closing_stock_t"), 0))
         .orderBy("month_start"))
display(check)
